# 12 — S.E.T.S Score

## Goal
Combine the five individual scores into a single **S.E.T.S** number and rate each zone.

---

## The Five Components

| Letter | Concept | Phase | Score |
|--------|---------|-------|-------|
| **S** | Strength (departure ATR-ratio) | NB06 | 0 – 2 |
| **T** | Time (base candle count)        | NB09 | 0 – 2 |
| **F** | Freshness (touch count)         | NB08 | 0 – 2 |
| **A** | Alignment (trend)               | NB11 | 0 – 2 |
| **C** | Curve (HTF position)            | NB10 | 0 – 2 |

$$\text{SETS} = S + T + F + A + C \quad \in [0, 10]$$

---

## Rating

| SETS total | Rating |
|------------|--------|
| ≥ 7        | ★★★  A-setup — take it |
| 5 – 6      | ★★   B-setup — trade with caution |
| ≤ 4        | ★    Skip |

## 1. Setup — load data and run the full detection pipeline

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import plotly.io as pio
pio.renderers.default = "notebook_connected"

from utils.data_loader     import load_all_timeframes
from utils.models          import CandlePrimitives, add_atr
from utils.base_detector   import detect_bases
from utils.legs_formation  import detect_formations
from utils.zone_detector   import detect_zones
from utils.freshness       import add_freshness
from utils.time_scoring    import add_time_score
from utils.htf_range       import add_curve_score
from utils.trend_alignment import add_trend_score
from utils.sets_scoring    import add_sets_score
from utils.config          import HTF_REF, HTF_RANGE_LOOKBACK, SWING_WINDOW

# ── load & enrich ──────────────────────────────────────────────────────────────
SYMBOL   = "USDJPY=X"
KEEP_TFS = ["1d", "4h", "1h"]
raw  = load_all_timeframes(SYMBOL, align=True)
data = {
    tf: add_atr(CandlePrimitives.enrich_dataframe(df))
    for tf, df in raw.items()
    if tf in KEEP_TFS
}
print("Loaded timeframes:", list(data.keys()))

# ── full pipeline for 4h and 1h ──────────────────────────────────────────────
results = {}

for tf in ["4h", "1h"]:
    df      = data[tf]
    htf_key = HTF_REF.get(tf, "1d")
    htf_df  = data[htf_key] if htf_key in data else data["1d"]

    passed_bases, _ = detect_bases(df)
    formations      = detect_formations(df, passed_bases)
    passed, _       = detect_zones(df, formations)

    add_freshness(df, passed)
    add_time_score(passed)
    add_curve_score(passed, htf_df, df.index, HTF_RANGE_LOOKBACK)
    add_trend_score(passed, df, SWING_WINDOW)
    add_sets_score(passed)

    results[tf] = {"df": df, "zones": passed}
    n_a = sum(1 for z in passed if z["sets_rating"] == "★★★")
    n_b = sum(1 for z in passed if z["sets_rating"] == "★★")
    n_s = sum(1 for z in passed if z["sets_rating"] == "★")
    print(f"[{tf}]  zones={len(passed):3d}  ★★★={n_a}  ★★={n_b}  ★={n_s}")


Loaded timeframes: ['1d', '4h', '1h']
[4h]  zones= 25  ★★★=5  ★★=12  ★=8
[1h]  zones= 97  ★★★=28  ★★=42  ★=27


## 2. Scores summary table

In [2]:
TF_SHOW = "4h"   # change to "1h" to see the other frame

df_show  = results[TF_SHOW]["df"]
df_zones = results[TF_SHOW]["zones"]

rows = []
for z in df_zones:
    bar_ts = df_show.index[z["start"]]
    rows.append({
        "date"      : bar_ts.strftime("%Y-%m-%d %H:%M"),
        "type"      : z["zone_type"],
        "formation" : z["formation"],
        "S"         : z["strength_score"],
        "T"         : z["time_score"],
        "F"         : z["freshness_score"],
        "A"         : z["trend_score"],
        "C"         : z["curve_score"],
        "SETS"      : z["sets_total"],
        "Rating"    : z["sets_rating"],
    })

df_summary = pd.DataFrame(rows).sort_values("SETS", ascending=False)
df_summary


,date,type,formation,S,T,F,A,C,SETS,Rating
9,2025-04-09 04:00,demand,DBR,1,2,1,2,2,8,★★★
24,2026-03-19 08:00,supply,DBD,2,2,0,1,2,7,★★★
23,2026-03-18 20:00,supply,RBD,1,2,0,2,2,7,★★★
4,2024-11-26 20:00,supply,DBD,1,2,0,2,2,7,★★★
15,2025-08-14 04:00,demand,DBR,2,2,0,2,1,7,★★★
18,2025-09-24 20:00,demand,RBR,1,2,0,2,1,6,★★
1,2024-07-25 04:00,demand,DBR,1,2,0,1,2,6,★★
12,2025-06-23 08:00,supply,RBD,2,2,0,0,2,6,★★
2,2024-08-02 16:00,supply,DBD,2,2,0,2,0,6,★★
5,2024-12-18 20:00,demand,RBR,2,2,0,2,0,6,★★


## 3. Visualization — SETS bar chart + A/B zones on price

In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from utils.config import COLOR_BULL, COLOR_BEAR, CHART_BG, CHART_GRID

TF_VIZ       = "4h"   # change to "1h" for the other frame
df_viz       = results[TF_VIZ]["df"]
zones_viz    = results[TF_VIZ]["zones"]
RATING_COLOR = {"★★★": "#26a69a", "★★": "#ffb74d", "★": "#ef5350"}

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=False,
    row_heights=[0.60, 0.40],
    subplot_titles=[f"Price — ★★★ / ★★ zones  ({TF_VIZ})", "S.E.T.S total per zone"],
    vertical_spacing=0.10,
)

# ── candlesticks ──────────────────────────────────────────────────────────────
fig.add_trace(
    go.Candlestick(
        x=df_viz.index,
        open=df_viz["open"], high=df_viz["high"],
        low=df_viz["low"],   close=df_viz["close"],
        increasing_line_color=COLOR_BULL,
        decreasing_line_color=COLOR_BEAR,
        name="Price",
    ),
    row=1, col=1,
)

# ── zone boxes ────────────────────────────────────────────────────────────────
for z in zones_viz:
    rating = z["sets_rating"]
    if rating == "★":
        continue
    x0     = df_viz.index[z["start"]]
    x1     = df_viz.index[z["end"]]
    c_edge = RATING_COLOR[rating]
    alpha  = 0.22 if rating == "★★★" else 0.12
    fill   = f"rgba(38,166,154,{alpha})" if z["zone_type"] == "demand" else f"rgba(239,83,80,{alpha})"
    fig.add_vrect(x0=x0, x1=x1, fillcolor=fill,
                  line=dict(color=c_edge, width=1.5), row=1, col=1)
    fig.add_annotation(
        x=x0, y=z["proximal"],
        text=f"{z['formation']} {rating} ({z['sets_total']})",
        showarrow=False, font=dict(size=8, color=c_edge),
        xanchor="left", yanchor="bottom",
        row=1, col=1,
    )

# ── bar chart ─────────────────────────────────────────────────────────────────
bar_labels = [df_viz.index[z["start"]].strftime("%m-%d %H:%M") for z in zones_viz]
bar_totals = [z["sets_total"] for z in zones_viz]
bar_colors = [RATING_COLOR[z["sets_rating"]] for z in zones_viz]

fig.add_trace(
    go.Bar(x=bar_labels, y=bar_totals, marker_color=bar_colors, name="SETS total"),
    row=2, col=1,
)
fig.add_hline(y=7, line=dict(color="#26a69a", dash="dash"),
              annotation_text="A (≥7)", annotation_position="right",
              row=2, col=1)
fig.add_hline(y=5, line=dict(color="#ffb74d", dash="dash"),
              annotation_text="B (≥5)", annotation_position="right",
              row=2, col=1)

# ── layout ────────────────────────────────────────────────────────────────────
fig.update_layout(
    title=f"S.E.T.S Scoring — {SYMBOL} {TF_VIZ}",
    height=700,
    plot_bgcolor=CHART_BG,
    paper_bgcolor=CHART_BG,
    font_color="white",
    xaxis_rangeslider_visible=False,
    hovermode="x unified",
    showlegend=False,
)
for ax in ["xaxis", "xaxis2", "yaxis", "yaxis2"]:
    fig.update_layout(**{ax: dict(gridcolor=CHART_GRID, showgrid=True,
                                  zeroline=False, linecolor=CHART_GRID)})
fig.show()


## What's next?

**NB13 — Nested Zones**  
Identify when a smaller zone sits inside a higher-timeframe zone, boosting confidence for entries that are aligned across multiple timeframes.